# Phase 1.5: Posture Merge

Read the posture xlsx (182 rows), standardise the columns, then merge into the cleaned survey by severity rank: sort survey by exposure severity (pain + fatigue + hours), sort posture by RULA Table C, pair rank-to-rank.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
CLEANED = ROOT / 'data' / 'processed' / 'cleaned.csv'
POSTURE = ROOT / 'data' / 'raw' / 'posture_data.xlsx'
OUT     = ROOT / 'data' / 'processed'

## Step 1 — Read survey and the posture xlsx

In [2]:
survey = pd.read_csv(CLEANED)
print('survey:', survey.shape)

# Standardise to a single posture frame. Drop the first column (image / blank)
# and rename the next 26 columns positionally so the data combines cleanly.
STD_COLS = [
    'upper_arm', 'lower_arm', 'wrist', 'wrist_twist',
    'neck_position', 'trunk_position', 'legs',
    'muscle_a', 'force_a', 'wrist_arm_score',
    'muscle_b', 'force_b', 'ntl_score',
    'rula_table_a', 'rula_table_b', 'rula_table_c',
    'col_17', 'col_18',
    'qec_back', 'qec_shoulder_arm', 'qec_wrist_hand', 'qec_neck',
    'qec_driving', 'qec_vibration', 'qec_work_pace', 'qec_stress',
]

frames = []
for sh in pd.ExcelFile(POSTURE).sheet_names:
    p = pd.read_excel(POSTURE, sheet_name=sh)
    p = p.iloc[:, 1:27]
    p.columns = STD_COLS
    p = p.dropna(subset=['rula_table_c']).reset_index(drop=True)
    frames.append(p)

posture = pd.concat(frames, ignore_index=True).drop(columns=['col_17', 'col_18'])
print('posture:', posture.shape)
posture.head()

survey: (182, 48)


posture: (182, 24)


,upper_arm,lower_arm,wrist,wrist_twist,neck_position,trunk_position,legs,muscle_a,force_a,wrist_arm_score,muscle_b,force_b,ntl_score,rula_table_a,rula_table_b,rula_table_c,qec_back,qec_shoulder_arm,qec_wrist_hand,qec_neck,qec_driving,qec_vibration,qec_work_pace,qec_stress
0,3,1,2,1,3,2,1,1.0,1,6,1,0,4,4,3,6,14,24,30,10,4,4,4,2
1,1,2,2,1,3,2,1,1.0,0,3,1,0,4,2,3,4,24,30,36,16,9,9,9,9
2,3,1,2,1,3,2,1,1.0,2,7,1,2,6,4,3,7,28,40,36,14,9,9,6,9
3,3,1,2,1,2,2,1,1.0,1,6,1,0,3,4,2,5,22,32,30,10,4,4,4,6
4,3,1,2,1,3,3,1,1.0,1,6,1,0,5,4,4,6,22,32,34,12,6,6,6,9


## Step 2 — Compute survey exposure severity (pain + fatigue + hours)

In [3]:
# Pain count: number of 'Yes' across the 9 Nordic body-area items
nordic_cols = [c for c in survey.columns if 'In the past 12 months' in c]
pain_count = (survey[nordic_cols] == 'Yes').sum(axis=1)

# Fatigue: mean of the 6 Borg CR10 items (last 6 columns, already 0-10)
borg_cols = list(survey.columns[-6:])
fatigue_mean = survey[borg_cols].mean(axis=1)

# Working hours -> midpoint number
hours_map = {'< 3 hrs': 2, '3-5 hrs': 4, '6-8 hrs': 7, '> 8 hrs': 9}
hours_num = survey['Working Hours per Day'].map(hours_map)

# Normalise each component to 0-1 so they contribute equally to the rank
exposure_severity = (
    pain_count   / pain_count.max()
  + fatigue_mean / fatigue_mean.max()
  + hours_num    / hours_num.max()
)

survey['pain_count']        = pain_count
survey['fatigue_mean']      = fatigue_mean.round(2)
survey['hours_num']         = hours_num
survey['exposure_severity'] = exposure_severity.round(3)

survey[['pain_count', 'fatigue_mean', 'hours_num', 'exposure_severity']].describe()

,pain_count,fatigue_mean,hours_num,exposure_severity
count,182.000000,182.000000,182.000000,182.000000
mean,3.956044,4.530549,7.346154,1.759181
std,2.772844,1.938360,1.965437,0.517550
min,0.000000,0.000000,2.000000,0.463000
25%,2.000000,3.170000,7.000000,1.411750
50%,4.000000,4.415000,7.000000,1.731500
75%,6.000000,5.957500,9.000000,2.106500
max,9.000000,9.000000,9.000000,2.963000


## Step 3 — Posture severity = RULA Table C

In [4]:
posture['posture_severity'] = posture['rula_table_c']

print('posture rows:', len(posture))
posture[['rula_table_a', 'rula_table_b', 'rula_table_c',
         'qec_back', 'qec_shoulder_arm', 'qec_wrist_hand', 'qec_neck',
         'qec_driving', 'qec_vibration', 'qec_work_pace', 'qec_stress']].describe()

posture rows: 182


,rula_table_a,rula_table_b,rula_table_c,qec_back,qec_shoulder_arm,qec_wrist_hand,qec_neck,qec_driving,qec_vibration,qec_work_pace,qec_stress
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000
mean,3.675824,3.219780,5.829670,26.967033,30.032967,30.758242,17.659341,7.879121,7.181319,5.549451,8.302198
std,0.630152,1.135098,1.160396,6.092202,8.081013,8.243317,5.655330,2.342663,2.213740,2.556602,4.285643
min,2.000000,1.000000,3.000000,14.000000,14.000000,14.000000,10.000000,4.000000,4.000000,1.000000,1.000000
25%,3.000000,2.000000,5.000000,24.000000,24.500000,26.000000,14.000000,6.000000,4.000000,4.000000,4.000000
50%,4.000000,3.000000,6.000000,28.000000,28.000000,28.000000,16.000000,9.000000,9.000000,4.000000,9.000000
75%,4.000000,4.000000,7.000000,30.000000,34.000000,37.500000,18.000000,9.000000,9.000000,8.000000,9.000000
max,5.000000,6.000000,7.000000,46.000000,56.000000,46.000000,36.000000,16.000000,9.000000,10.000000,16.000000


## Step 4 — Sort both by severity and pair rank-to-rank

In [5]:
assert len(survey) == len(posture), f'survey={len(survey)} vs posture={len(posture)}'

survey_sorted = survey.sort_values('exposure_severity', ascending=False).reset_index(drop=True)
survey_sorted['rank'] = survey_sorted.index + 1

posture_sorted = posture.sort_values('posture_severity', ascending=False).reset_index(drop=True)
posture_sorted['rank'] = posture_sorted.index + 1

posture_cols = ['upper_arm', 'lower_arm', 'wrist', 'wrist_twist',
                'neck_position', 'trunk_position', 'legs',
                'muscle_a', 'force_a', 'muscle_b', 'force_b',
                'rula_table_a', 'rula_table_b', 'rula_table_c',
                'qec_back', 'qec_shoulder_arm', 'qec_wrist_hand', 'qec_neck',
                'qec_driving', 'qec_vibration', 'qec_work_pace', 'qec_stress',
                'posture_severity', 'rank']

merged = survey_sorted.merge(posture_sorted[posture_cols], on='rank', suffixes=('', '_p'))

print('merged shape:', merged.shape)
merged[['rank', 'exposure_severity', 'rula_table_c',
        'qec_back', 'qec_neck', 'qec_driving', 'qec_stress']].head(10)

merged shape: (182, 76)


,rank,exposure_severity,rula_table_c,qec_back,qec_neck,qec_driving,qec_stress
0,1,2.963,7,20,16,9,4
1,2,2.944,7,28,14,9,9
2,3,2.926,7,30,20,16,8
3,4,2.907,7,28,18,9,9
4,5,2.889,7,26,14,9,9
5,6,2.870,7,28,16,9,9
6,7,2.833,7,18,20,4,2
7,8,2.815,7,30,28,8,16
8,9,2.815,7,28,28,8,16
9,10,2.778,7,28,28,8,16


## Step 5 — Save

In [6]:
out = OUT / 'cleaned_with_posture.csv'
merged.to_csv(out, index=False)
print('saved', out, merged.shape)

saved f:\Ergonomic_Project\data\processed\cleaned_with_posture.csv (182, 76)
